## Cell 1 - Note

In [1]:
# ============================================================
# 03_supervised_source_train.ipynb
# Hybrid supervised training
#   phase/AoA  -> geometry branch
#   amplitude  -> confidence / correction branch
# ============================================================
#
# Inputs:
#   - train_labeled.npz
#   - pred_support.npz   (optional for quick adaptation-style validation)
#   - amp_ssl_encoder_best.keras
#
# Outputs:
#   - hybrid_model_best.keras
#   - hybrid_model_final.keras
#   - hybrid_train_history.json
#   - class_centers.json
# ============================================================

## Cell 2 - Import

In [2]:
import os
import json
import math
import random
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

2026-04-06 16:12:54.861211: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-06 16:12:54.867466: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775491974.874848 2821320 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775491974.877098 2821320 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775491974.882749 2821320 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

## Cell 3 - Config

In [3]:
# ============================================================
# CONFIG
# ============================================================

DATA_ROOT = Path("/home/tonyliao/Location")   # change this
BUILD_DIR = DATA_ROOT / "dataset_build_hybrid"
SSL_DIR   = DATA_ROOT / "ssl_pretrain_runs"
OUT_DIR   = DATA_ROOT / "hybrid_train_runs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_LABELED_NPZ = BUILD_DIR / "train_labeled.npz"
PRED_SUPPORT_NPZ  = BUILD_DIR / "pred_support.npz"
LABEL_MAP_JSON    = BUILD_DIR / "label_map.json"

AMP_SSL_ENCODER_PATH = SSL_DIR / "amp_ssl_encoder_best.keras"

USE_AVECSI = True
USE_PHASE  = True

USE_PRED_SUPPORT_AS_VAL = False
VAL_RATIO = 0.2
SEED = 42

# input resize
TARGET_T = 256
TARGET_S = None   # infer from first sample

# train
BATCH_SIZE = 16
EPOCHS = 100
LR = 1e-4
WEIGHT_DECAY = 1e-5

# losses
W_PRES = 0.5
W_CLS  = 1.0
W_AOA  = 1.0
W_DELTA = 0.5
W_FINAL = 1.5

# amplitude correction clamp
DELTA_LIMIT = 1.5   # meters, approximate correction bound

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## Cell 4 - GPU Enable

In [4]:
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as e:
        print("memory growth warning:", e)

TensorFlow: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Cell 5 - Load Data

In [5]:
train_obj = np.load(TRAIN_LABELED_NPZ, allow_pickle=True)
print("train labeled n =", len(train_obj["label_id"]))

pred_support_obj = None
if USE_PRED_SUPPORT_AS_VAL and PRED_SUPPORT_NPZ.exists():
    pred_support_obj = np.load(PRED_SUPPORT_NPZ, allow_pickle=True)
    print("pred support n =", len(pred_support_obj["label_id"]))

with open(LABEL_MAP_JSON, "r", encoding="utf-8") as f:
    label_meta = json.load(f)

label_map = label_meta["label_map"]
inv_label_map = {int(k): v for k, v in label_meta["inv_label_map"].items()}
print(label_map)

train labeled n = 1774
{'Empty': 0, 'LeftDown': 1, 'LeftMid': 2, 'LeftUp': 3, 'MiddleDown': 4, 'MiddleUp': 5, 'RightDown': 6, 'RightMid': 7, 'RightUp': 8}


## Cell 6 - Helper

In [6]:
def np_load_float32(path):
    return np.load(str(path)).astype(np.float32)

def ensure_3d(x):
    x = np.asarray(x, dtype=np.float32)
    if x.ndim == 2:
        x = x[..., None]
    return x

def resize_to_target(x, target_t, target_s):
    x_tf = tf.convert_to_tensor(x, dtype=tf.float32)
    x_tf = tf.image.resize(x_tf, size=(target_t, target_s), method="bilinear")
    return x_tf.numpy().astype(np.float32)

def zscore_per_sample(x):
    mu = np.mean(x, axis=(0,1), keepdims=True)
    sd = np.std(x, axis=(0,1), keepdims=True) + 1e-6
    return ((x - mu) / sd).astype(np.float32)

def infer_target_s(npz_obj):
    a_amp = np_load_float32(npz_obj["A_amp_paths"][0])
    a_amp = ensure_3d(a_amp)
    return a_amp.shape[1]

if TARGET_S is None:
    TARGET_S = infer_target_s(train_obj)

print("TARGET_T =", TARGET_T)
print("TARGET_S =", TARGET_S)

def build_amp_input_from_row(npz_obj, idx, use_avecsi=True):
    feats = []

    A = ensure_3d(np_load_float32(npz_obj["A_amp_paths"][idx]))
    B = ensure_3d(np_load_float32(npz_obj["B_amp_paths"][idx]))
    feats += [A, B]

    if use_avecsi:
        aavg_path = str(npz_obj["A_ampavg_paths"][idx])
        bavg_path = str(npz_obj["B_ampavg_paths"][idx])

        if aavg_path and bavg_path:
            Aavg = ensure_3d(np_load_float32(aavg_path))
            Bavg = ensure_3d(np_load_float32(bavg_path))

            Tref = A.shape[0]
            if Aavg.shape[0] == 1 and Tref > 1:
                Aavg = np.repeat(Aavg, Tref, axis=0)
            if Bavg.shape[0] == 1 and Tref > 1:
                Bavg = np.repeat(Bavg, Tref, axis=0)

            feats += [Aavg, Bavg]

    T0, S0 = feats[0].shape[:2]
    out = []
    for x in feats:
        if x.shape[0] != T0 or x.shape[1] != S0:
            x = resize_to_target(x, T0, S0)
        out.append(x)

    x = np.concatenate(out, axis=-1)
    x = resize_to_target(x, TARGET_T, TARGET_S)
    x = zscore_per_sample(x)
    return x.astype(np.float32)

def build_phase_input_from_row(npz_obj, idx):
    A = ensure_3d(np_load_float32(npz_obj["A_pha_paths"][idx]))
    B = ensure_3d(np_load_float32(npz_obj["B_pha_paths"][idx]))

    T0 = max(A.shape[0], B.shape[0])
    S0 = max(A.shape[1], B.shape[1])

    if A.shape[:2] != (T0, S0):
        A = resize_to_target(A, T0, S0)
    if B.shape[:2] != (T0, S0):
        B = resize_to_target(B, T0, S0)

    x = np.concatenate([A, B], axis=-1)
    x = resize_to_target(x, TARGET_T, TARGET_S)
    x = zscore_per_sample(x)
    return x.astype(np.float32)

def collect_rows_from_npz(npz_obj):
    rows = []
    n = len(npz_obj["label_id"])
    for i in range(n):
        rows.append(i)
    return rows

train_indices = collect_rows_from_npz(train_obj)
print("train rows:", len(train_indices))

TARGET_T = 256
TARGET_S = 41
train rows: 1774


## Cell 7 - Normalize Label 

In [7]:
# Normalize labels to 2D anchor coordinates
# Change these coordinates to your real layout if you have measured positions.

CLASS_CENTER_MAP = {
    "Empty":       [0.0, 0.0],
    "LeftDown":    [2.0, 6.0],
    "LeftMid":     [2.0, 4.0],
    "LeftUp":      [2.0, 2.0],
    "MiddleDown":  [3.0, 6.0],
    "MiddleUp":    [3.0, 2.0],
    "RightDown":   [4.0, 6.0],
    "RightMid":    [4.0, 4.0],
    "RightUp":     [4.0, 2.0],
}

# Empty can be ignored in localization loss by masking later if needed.
print(CLASS_CENTER_MAP)

with open(OUT_DIR / "class_centers.json", "w", encoding="utf-8") as f:
    json.dump(CLASS_CENTER_MAP, f, indent=2)
print("saved class centers")

def get_xy_from_label_name(label_name):
    xy = CLASS_CENTER_MAP[str(label_name)]
    return np.asarray(xy, dtype=np.float32)

def onehot(num_classes, y):
    v = np.zeros((num_classes,), dtype=np.float32)
    v[int(y)] = 1.0
    return v

num_classes = len(label_map)
print("num_classes =", num_classes)

sample_amp = build_amp_input_from_row(train_obj, 0, use_avecsi=USE_AVECSI)
sample_pha = build_phase_input_from_row(train_obj, 0)

print("sample_amp:", sample_amp.shape, sample_amp.dtype)
print("sample_pha:", sample_pha.shape, sample_pha.dtype)

{'Empty': [0.0, 0.0], 'LeftDown': [2.0, 6.0], 'LeftMid': [2.0, 4.0], 'LeftUp': [2.0, 2.0], 'MiddleDown': [3.0, 6.0], 'MiddleUp': [3.0, 2.0], 'RightDown': [4.0, 6.0], 'RightMid': [4.0, 4.0], 'RightUp': [4.0, 2.0]}
saved class centers
num_classes = 9
sample_amp: (256, 41, 8) float32
sample_pha: (256, 41, 4) float32


I0000 00:00:1775491976.165727 2821320 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22181 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9


## Cell 8 - Split - Train/Val

In [8]:
def split_train_indices(npz_obj, val_ratio=0.2, seed=42):
    y = npz_obj["label_id"].astype(np.int64)
    idx_all = np.arange(len(y))
    idx_tr, idx_va = train_test_split(
        idx_all,
        test_size=val_ratio,
        random_state=seed,
        stratify=y,
    )
    return idx_tr, idx_va

if USE_PRED_SUPPORT_AS_VAL and pred_support_obj is not None:
    train_idx = np.arange(len(train_obj["label_id"]))
    val_source = "pred_support"
else:
    train_idx, val_idx = split_train_indices(train_obj, val_ratio=VAL_RATIO, seed=SEED)
    val_source = "train_split"

print("val_source:", val_source)
print("train_idx:", len(train_idx))
if val_source == "train_split":
    print("val_idx:", len(val_idx))

val_source: train_split
train_idx: 1419
val_idx: 355


## Cell 9 - Class Hybird Define

In [9]:
class HybridSequence(keras.utils.Sequence):
    def __init__(self, npz_obj, indices, batch_size=16, shuffle=True):
        self.npz_obj = npz_obj
        self.indices = np.array(indices, dtype=np.int64)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        return math.ceil(len(self.indices) / self.batch_size)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        inds = self.indices[idx*self.batch_size:(idx+1)*self.batch_size]

        x_amp = []
        x_pha = []

        y_presence = []
        y_cls = []
        y_xy = []
        y_aoa = []
        y_delta = []
        sample_w = []

        for i in inds:
            amp = build_amp_input_from_row(self.npz_obj, i, use_avecsi=USE_AVECSI)
            pha = build_phase_input_from_row(self.npz_obj, i)

            label_id = int(self.npz_obj["label_id"][i])
            label_name = str(self.npz_obj["label_name"][i])
            presence = int(self.npz_obj["presence"][i])

            xy = get_xy_from_label_name(label_name)

            # Supervision target design:
            # - aoa target = same anchor xy for now
            # - delta target = zero initially (safe baseline)
            aoa_xy = xy.copy()
            delta_xy = np.zeros((2,), dtype=np.float32)

            # optional: lower weight for empty
            w = 0.25 if presence == 0 else 1.0

            x_amp.append(amp)
            x_pha.append(pha)

            y_presence.append([presence])
            y_cls.append(onehot(num_classes, label_id))
            y_xy.append(xy)
            y_aoa.append(aoa_xy)
            y_delta.append(delta_xy)
            sample_w.append(w)

        x = {
            "amp_in": np.stack(x_amp).astype(np.float32),
            "pha_in": np.stack(x_pha).astype(np.float32),
        }

        y = {
            "presence_out": np.stack(y_presence).astype(np.float32),
            "class_out": np.stack(y_cls).astype(np.float32),
            "aoa_xy_out": np.stack(y_aoa).astype(np.float32),
            "amp_delta_out": np.stack(y_delta).astype(np.float32),
            "final_xy_out": np.stack(y_xy).astype(np.float32),
        }

        sw = {
            "presence_out": np.asarray(sample_w, dtype=np.float32),
            "class_out": np.asarray(sample_w, dtype=np.float32),
            "aoa_xy_out": np.asarray(sample_w, dtype=np.float32),
            "amp_delta_out": np.asarray(sample_w, dtype=np.float32),
            "final_xy_out": np.asarray(sample_w, dtype=np.float32),
        }

        return x, y, sw

## Cell 10 - Test Data Generator

In [10]:
train_gen = HybridSequence(train_obj, train_idx, batch_size=BATCH_SIZE, shuffle=True)

if val_source == "train_split":
    val_gen = HybridSequence(train_obj, val_idx, batch_size=BATCH_SIZE, shuffle=False)
else:
    pred_idx = np.arange(len(pred_support_obj["label_id"]))
    val_gen = HybridSequence(pred_support_obj, pred_idx, batch_size=BATCH_SIZE, shuffle=False)

bx, by, bsw = train_gen[0]
print("amp_in:", bx["amp_in"].shape)
print("pha_in:", bx["pha_in"].shape)
for k, v in by.items():
    print(k, v.shape)

amp_in: (16, 256, 41, 8)
pha_in: (16, 256, 41, 4)
presence_out (16, 1)
class_out (16, 9)
aoa_xy_out (16, 2)
amp_delta_out (16, 2)
final_xy_out (16, 2)


## Cell 11 - Conv & Build Phase Define

In [11]:
def conv_block_2d(x, filters, pool=True, dropout=0.0):
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    if pool:
        x = layers.MaxPooling2D(pool_size=2)(x)

    if dropout > 0:
        x = layers.Dropout(dropout)(x)

    return x

def build_phase_geometry_encoder(input_shape):
    inp = keras.Input(shape=input_shape, name="pha_geom_input")

    x = conv_block_2d(inp, 32, pool=True, dropout=0.05)
    x = conv_block_2d(x, 64, pool=True, dropout=0.05)
    x = conv_block_2d(x, 128, pool=True, dropout=0.10)
    x = conv_block_2d(x, 256, pool=True, dropout=0.10)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    feat = layers.Dense(128, activation="relu", name="pha_feat")(x)

    model = keras.Model(inp, feat, name="phase_geometry_encoder")
    return model

In [12]:
# load SSL-pretrained amplitude encoder
amp_encoder = keras.models.load_model(AMP_SSL_ENCODER_PATH, compile=False)
print("Loaded amp encoder from:", AMP_SSL_ENCODER_PATH)
amp_encoder.summary()

Loaded amp encoder from: /home/tonyliao/Location/ssl_pretrain_runs/amp_ssl_encoder_best.keras


Model: "amp_ssl_encoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ amp_ssl_input (InputLayer)      │ (None, 256, 41, 8)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 256, 41, 32)    │         2,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256, 41, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 256, 41, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 256, 41, 32)    │         9,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256, 41, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 256, 41, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 128, 20, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128, 20, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 128, 20, 64)    │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128, 20, 64)    │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 128, 20, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 128, 20, 64)    │        36,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 128, 20, 64)    │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_3 (ReLU)                  │ (None, 128, 20, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 64, 10, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64, 10, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 64, 10, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 64, 10, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_4 (ReLU)                  │ (None, 64, 10, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 64, 10, 128)    │       147,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 64, 10, 128)    │           512 │
│ (BatchNormalization)            │                        │             

 Total params: 1,308,160 (4.99 MB)

 Trainable params: 1,306,240 (4.98 MB)

 Non-trainable params: 1,920 (7.50 KB)

In [13]:
def build_hybrid_model(amp_input_shape, pha_input_shape, amp_encoder, num_classes):
    amp_in = keras.Input(shape=amp_input_shape, name="amp_in")
    pha_in = keras.Input(shape=pha_input_shape, name="pha_in")

    # amplitude branch
    amp_feat = amp_encoder(amp_in)
    amp_feat = layers.Dense(128, activation="relu", name="amp_feat")(amp_feat)

    # phase / geometry branch
    pha_encoder = build_phase_geometry_encoder(pha_input_shape)
    pha_feat = pha_encoder(pha_in)

    # presence head from amplitude
    presence_out = layers.Dense(1, activation="sigmoid", name="presence_out")(amp_feat)

    # class head from fused representation
    fused_cls = layers.Concatenate()([amp_feat, pha_feat])
    fused_cls = layers.Dense(128, activation="relu")(fused_cls)
    class_out = layers.Dense(num_classes, activation="softmax", name="class_out")(fused_cls)

    # aoa geometry xy
    geo = layers.Dense(128, activation="relu")(pha_feat)
    aoa_xy_out = layers.Dense(2, activation="linear", name="aoa_xy_out")(geo)

    # amplitude correction
    corr = layers.Dense(128, activation="relu")(amp_feat)
    raw_delta = layers.Dense(2, activation="tanh")(corr)
    amp_delta_out = layers.Rescaling(scale=DELTA_LIMIT, name="amp_delta_out")(raw_delta)
    # correction gate
    gate_in = layers.Concatenate()([amp_feat, pha_feat, presence_out, class_out])
    gate = layers.Dense(64, activation="relu")(gate_in)
    gate = layers.Dense(1, activation="sigmoid", name="delta_gate")(gate)

    # final xy
    scaled_delta = layers.Multiply()([amp_delta_out, gate])
    final_xy_out = layers.Add(name="final_xy_out")([aoa_xy_out, scaled_delta])

    model = keras.Model(
        inputs={"amp_in": amp_in, "pha_in": pha_in},
        outputs={
            "presence_out": presence_out,
            "class_out": class_out,
            "aoa_xy_out": aoa_xy_out,
            "amp_delta_out": amp_delta_out,
            "final_xy_out": final_xy_out,
        },
        name="hybrid_aoa_amp_model"
    )
    return model

In [14]:
amp_input_shape = bx["amp_in"].shape[1:]
pha_input_shape = bx["pha_in"].shape[1:]

model = build_hybrid_model(
    amp_input_shape=amp_input_shape,
    pha_input_shape=pha_input_shape,
    amp_encoder=amp_encoder,
    num_classes=num_classes,
)

model.summary()

Model: "hybrid_aoa_amp_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ amp_in (InputLayer) │ (None, 256, 41,   │          0 │ -                 │
│                     │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ amp_ssl_encoder     │ (None, 256)       │  1,308,160 │ amp_in[0][0]      │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pha_in (InputLayer) │ (None, 256, 41,   │          0 │ -                 │
│                     │ 4)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ amp_feat (Dense)    │ (None, 128)       │     32,896 │ amp_ssl_encoder[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ phase_geometry_enc… │ (None, 128)       │  1,274,112 │ pha_in[0][0]      │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 256)       │          0 │ amp_feat[0][0],   │
│ (Concatenate)       │                   │            │ phase_geometry_e… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     32,896 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ class_out (Dense)   │ (None, 9)         │      1,161 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ presence_out        │ (None, 1)         │        129 │ amp_feat[0][0]    │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 128)       │     16,512 │ amp_feat[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 266)       │          0 │ amp_feat[0][0],   │
│ (Concatenate)       │                   │            │ phase_geometry_e… │
│                     │                   │            │ presence_out[0][… │
│                     │                   │            │ class_out[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 2)         │        258 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 64)        │     17,088 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ amp_delta_out       │ (None, 2)         │          0 │ dense_4[0][0]     │
│ (Rescaling)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │     16,512 │ phase_geometry_e… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ delta_gate (Dense)  │ (None, 1)         │         65 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ aoa_xy_out (Dense)  │ (None, 2)         │        258 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 2)         │          0 │ amp_delta_out[0]… │
│                     │                   │            │ delta_gate[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ final_xy_out (Add)  │ (None, 2)         │          0 │ aoa_xy_out[0][0], │
│                     │                   │            │ multiply[0][0]  

 Total params: 2,700,047 (10.30 MB)

 Trainable params: 2,696,207 (10.29 MB)

 Non-trainable params: 3,840 (15.00 KB)

In [15]:
# Freeze most of amplitude encoder first
for layer in amp_encoder.layers:
    layer.trainable = False

# optionally unfreeze last few layers
for layer in amp_encoder.layers[-4:]:
    layer.trainable = True

print("Trainable layers in amp_encoder:")
for layer in amp_encoder.layers[-8:]:
    print(layer.name, layer.trainable)

Trainable layers in amp_encoder:
conv2d_7 False
batch_normalization_7 False
re_lu_7 False
max_pooling2d_3 False
dropout_3 True
global_average_pooling2d True
dense True
embedding True


In [16]:
losses = {
    "presence_out": keras.losses.BinaryCrossentropy(),
    "class_out": keras.losses.CategoricalCrossentropy(),
    "aoa_xy_out": keras.losses.Huber(),
    "amp_delta_out": keras.losses.Huber(),
    "final_xy_out": keras.losses.Huber(),
}

loss_weights = {
    "presence_out": W_PRES,
    "class_out": W_CLS,
    "aoa_xy_out": W_AOA,
    "amp_delta_out": W_DELTA,
    "final_xy_out": W_FINAL,
}

metrics = {
    "presence_out": [keras.metrics.BinaryAccuracy(name="acc")],
    "class_out": [keras.metrics.CategoricalAccuracy(name="acc")],
    "aoa_xy_out": [keras.metrics.MeanAbsoluteError(name="mae")],
    "amp_delta_out": [keras.metrics.MeanAbsoluteError(name="mae")],
    "final_xy_out": [keras.metrics.MeanAbsoluteError(name="mae")],
}

opt = keras.optimizers.Adam(learning_rate=LR)

model.compile(
    optimizer=opt,
    loss=losses,
    loss_weights=loss_weights,
    metrics=metrics,
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(OUT_DIR / "hybrid_model_best.keras"),
        monitor="val_final_xy_out_mae",
        save_best_only=True,
        mode="min",
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_final_xy_out_mae",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        mode="min",
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_final_xy_out_mae",
        patience=10,
        mode="min",
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.CSVLogger(str(OUT_DIR / "hybrid_train_log.csv")),
]

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

model.save(OUT_DIR / "hybrid_model_final.keras")

with open(OUT_DIR / "hybrid_train_history.json", "w", encoding="utf-8") as f:
    json.dump(history.history, f, indent=2)

print("Saved to:", OUT_DIR)

/home/tonyliao/tensorflow_env/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/100


I0000 00:00:1775491979.386319 2821454 service.cc:152] XLA service 0x7f51e003a460 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775491979.386336 2821454 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4090, Compute Capability 8.9
2026-04-06 16:12:59.472975: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1775491979.976928 2821454 cuda_dnn.cc:529] Loaded cuDNN version 90300
2026-04-06 16:13:01.294589: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_3172', 4 bytes spill stores, 4 bytes spill loads



 1/89 ━━━━━━━━━━━━━━━━━━━━ 16:44 11s/step - amp_delta_out_loss: 0.6127 - amp_delta_out_mae: 1.1552 - aoa_xy_out_loss: 3.1195 - aoa_xy_out_mae: 3.6544 - class_out_acc: 0.2500 - class_out_loss: 3.3860 - final_xy_out_loss: 3.1851 - final_xy_out_mae: 3.7460 - loss: 14.4190 - presence_out_acc: 0.1250 - presence_out_loss: 5.6589

I0000 00:00:1775491988.464769 2821454 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2026-04-06 16:13:10.178044: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_3172', 8 bytes spill stores, 8 bytes spill loads



89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 202ms/step - amp_delta_out_loss: 0.6359 - amp_delta_out_mae: 1.1338 - aoa_xy_out_loss: 1.3448 - aoa_xy_out_mae: 1.7541 - class_out_acc: 0.4091 - class_out_loss: 2.0477 - final_xy_out_loss: 1.2119 - final_xy_out_mae: 1.6206 - loss: 6.4023 - presence_out_acc: 0.5497 - presence_out_loss: 1.8977

2026-04-06 16:13:30.453968: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_363', 8 bytes spill stores, 8 bytes spill loads




Epoch 1: val_final_xy_out_mae improved from inf to 2.40681, saving model to /home/tonyliao/Location/hybrid_train_runs/hybrid_model_best.keras
89/89 ━━━━━━━━━━━━━━━━━━━━ 35s 272ms/step - amp_delta_out_loss: 0.6354 - amp_delta_out_mae: 1.1332 - aoa_xy_out_loss: 1.3366 - aoa_xy_out_mae: 1.7452 - class_out_acc: 0.4121 - class_out_loss: 2.0371 - final_xy_out_loss: 1.2042 - final_xy_out_mae: 1.6121 - loss: 6.3655 - presence_out_acc: 0.5522 - presence_out_loss: 1.8845 - val_amp_delta_out_loss: 0.3817 - val_amp_delta_out_mae: 0.8218 - val_aoa_xy_out_loss: 1.9682 - val_aoa_xy_out_mae: 2.4734 - val_class_out_acc: 0.6817 - val_class_out_loss: 1.0822 - val_final_xy_out_loss: 1.9032 - val_final_xy_out_mae: 2.4068 - val_loss: 6.1266 - val_presence_out_acc: 0.9380 - val_presence_out_loss: 0.0748 - learning_rate: 1.0000e-04
Epoch 2/100
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step - amp_delta_out_loss: 0.2793 - amp_delta_out_mae: 0.6777 - aoa_xy_out_loss: 0.0376 - aoa_xy_out_mae: 0.2091 - class_out_acc: 0

In [17]:
best_model = keras.models.load_model(
    OUT_DIR / "hybrid_model_best.keras",
    compile=False
)
print("Loaded best model.")
best_model.summary()

Loaded best model.


Model: "hybrid_aoa_amp_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ amp_in (InputLayer) │ (None, 256, 41,   │          0 │ -                 │
│                     │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ amp_ssl_encoder     │ (None, 256)       │  1,308,160 │ amp_in[0][0]      │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pha_in (InputLayer) │ (None, 256, 41,   │          0 │ -                 │
│                     │ 4)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ amp_feat (Dense)    │ (None, 128)       │     32,896 │ amp_ssl_encoder[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ phase_geometry_enc… │ (None, 128)       │  1,274,112 │ pha_in[0][0]      │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 256)       │          0 │ amp_feat[0][0],   │
│ (Concatenate)       │                   │            │ phase_geometry_e… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     32,896 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ class_out (Dense)   │ (None, 9)         │      1,161 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ presence_out        │ (None, 1)         │        129 │ amp_feat[0][0]    │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 128)       │     16,512 │ amp_feat[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 266)       │          0 │ amp_feat[0][0],   │
│ (Concatenate)       │                   │            │ phase_geometry_e… │
│                     │                   │            │ presence_out[0][… │
│                     │                   │            │ class_out[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 2)         │        258 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 64)        │     17,088 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ amp_delta_out       │ (None, 2)         │          0 │ dense_4[0][0]     │
│ (Rescaling)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │     16,512 │ phase_geometry_e… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ delta_gate (Dense)  │ (None, 1)         │         65 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ aoa_xy_out (Dense)  │ (None, 2)         │        258 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 2)         │          0 │ amp_delta_out[0]… │
│                     │                   │            │ delta_gate[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ final_xy_out (Add)  │ (None, 2)         │          0 │ aoa_xy_out[0][0], │
│                     │                   │            │ multiply[0][0]  

 Total params: 2,700,047 (10.30 MB)

 Trainable params: 1,521,551 (5.80 MB)

 Non-trainable params: 1,178,496 (4.50 MB)

In [18]:
def predict_one(npz_obj, idx, model):
    amp = build_amp_input_from_row(npz_obj, idx, use_avecsi=USE_AVECSI)
    pha = build_phase_input_from_row(npz_obj, idx)

    x = {
        "amp_in": np.expand_dims(amp, axis=0),
        "pha_in": np.expand_dims(pha, axis=0),
    }

    pred = model.predict(x, verbose=0)

    out = {
        "presence": float(pred["presence_out"][0,0]),
        "class_prob": pred["class_out"][0],
        "class_id": int(np.argmax(pred["class_out"][0])),
        "aoa_xy": pred["aoa_xy_out"][0],
        "amp_delta": pred["amp_delta_out"][0],
        "final_xy": pred["final_xy_out"][0],
    }
    return out

example = predict_one(train_obj, 0, best_model)
for k, v in example.items():
    print(k, v)

presence 0.9996334314346313
class_prob [5.5762416e-06 1.1894729e-06 2.4556148e-06 3.8550393e-06 6.7124193e-07
 9.9997509e-01 2.0125009e-07 1.0949204e-06 9.8428727e-06]
class_id 5
aoa_xy [2.893932  1.9707713]
amp_delta [ 0.01709298 -0.08697077]
final_xy [2.893986  1.9704974]


In [19]:
summary = {
    "train_labeled_npz": str(TRAIN_LABELED_NPZ),
    "pred_support_npz": str(PRED_SUPPORT_NPZ),
    "amp_ssl_encoder_path": str(AMP_SSL_ENCODER_PATH),
    "use_pred_support_as_val": USE_PRED_SUPPORT_AS_VAL,
    "target_t": TARGET_T,
    "target_s": TARGET_S,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "lr": LR,
    "weights": {
        "presence": W_PRES,
        "class": W_CLS,
        "aoa": W_AOA,
        "delta": W_DELTA,
        "final": W_FINAL,
    },
    "delta_limit": DELTA_LIMIT,
    "num_classes": num_classes,
    "seed": SEED,
}

with open(OUT_DIR / "hybrid_train_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Summary saved.")

Summary saved.


In [ ]:
import tensorflow as tf
tf.keras.backend.clear_session()
#Restart the kernel to free memory
import IPython
app = IPython.get_ipython()
app.kernel.do_shutdown(True)  # True = restart, False = shutdown

{'status': 'ok', 'restart': True}

: 